# Introduction to dyadic scales
Standardized fractional octave bands have geometric intervals. When these geometric intervals are base 2 [dyadic], they are a good match to computing systems in general and FFTs in particular. Since we will be performing many logarithmic operations, it is useful to have a measure of the smallest system number (epsilon) to preempt division by zero.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from quantum_inferno import qi_debugger
import quantum_inferno.scales_dyadic as sd
print('Module loaded. Epsilon:', sd.get_epsilon())

## Order and Cycles
Verify `scale_order_check`, `scale_multiplier`, `cycles_from_order`, and `order_from_cycles`.

In [ ]:
# Validate scale order sanitation
orders = [0.5, 0.75, 1.0, 3.0, 6.0, 12.0, -24.0]
sanitized = [sd.scale_order_check(o, show_warning=False) for o in orders]
print('Input orders:', orders)
print('Sanitized orders:', sanitized)
assert all(o >= sd.DEFAULT_SCALE_ORDER_MIN for o in sanitized)

# Cycles <-> Order consistency
for o in [1.0, 3.0, 6.0, 12.0, 24.0]:
    m = sd.cycles_from_order(o)
    o_back = sd.order_from_cycles(m)
    print(f'o={o}, cycles={m:.6f}, back={o_back:.6f}')
    assert np.isclose(o_back, sd.scale_order_check(o))

## Band Intervals (Periods)
Evaluate dyadic band edges and centers for a typical configuration.

In [ ]:
# Configuration
N = sd.Slice.ORD6
G = sd.Slice.G3
ref_s = 1.0
low_s = 0.01
high_s = 1.0
(scale_order, scale_base, n_bands, ref, center_alg, center_geo, start_s, end_s) = sd.band_intervals_periods(
            N, G, ref_s, low_s, high_s, show_warnings=False
        )
print('Bands:', n_bands[:5], '...', n_bands[-5:])
print('Counts:', len(n_bands))
print('First/last center (s):', center_geo[0], center_geo[-1])
# Plot period bands
plt.figure(figsize=(6,3))
plt.plot(n_bands, center_geo, 'o-', label='Geometric center (s)')
plt.xlabel('Band number n')
plt.ylabel('Center period (s)')
plt.grid(True)
plt.legend(); plt.tight_layout()

## Frequency Bands from FFT Points
Replicate expectations similar to `tests/test_scales_dyadic.py`: for a 100 Hz sample and 60 s display, dyadic bands should match expected length and edge values.

In [ ]:
# Parameters mirroring the test case
scale_order0 = 6.0
frequency_sample_hz0 = 100.0
time_display_points_float = sd.DEFAULT_TIME_DISPLAY_S * frequency_sample_hz0
time_display_points_pow2 = int(2 ** np.ceil(np.log2(time_display_points_float)))
physical_frequency_hz = sd.log_frequency_hz_from_fft_points(
            frequency_sample_hz0, time_display_points_pow2,
            scale_order0, sd.Slice.F1HZ, sd.Slice.G3
        )
print('time_display_points_float:', time_display_points_float)
print('time_display_points_pow2:', time_display_points_pow2)
print('log2(pow2):', np.log2(time_display_points_pow2))
print('first freq:', physical_frequency_hz[0])
print('last freq:', physical_frequency_hz[-1])
print('len(freqs):', len(physical_frequency_hz))
# Assertions based on expected values (tolerances for numerical stability)
assert time_display_points_float == 6000.0
assert time_display_points_pow2 == 8192
assert np.isclose(np.log2(time_display_points_pow2), 13.0)
assert np.isclose(physical_frequency_hz[0], 0.1778279410038923, rtol=1e-12, atol=1e-12)
assert np.isclose(physical_frequency_hz[-1], 39.810717055349706, rtol=1e-12, atol=1e-12)
assert len(physical_frequency_hz) == 48
# Plot the frequency bands
plt.figure(figsize=(6,3))
plt.plot(np.arange(len(physical_frequency_hz)), physical_frequency_hz, 'o-')
plt.xlabel('Band index')
plt.ylabel('Frequency (Hz)')
plt.grid(True); plt.tight_layout()

## Frequency Range Conversion
Evaluate `band_frequency_low_high` using a typical audio range to get frequency band edges and centers.

In [ ]:
N = sd.Slice.ORD6
G = sd.Slice.G3
f_ref = sd.Slice.F1HZ
f_low = 20.0   # Hz
f_high = 20000.0  # Hz
fs = 48000.0  # Hz
(N_out, G_out, band_nums, f_ref_out, f_center_alg, f_center_geo, f_start, f_end) = sd.band_frequency_low_high(
            N, G, f_ref, f_low, f_high, fs
        )
print('Order/Base:', N_out, G_out)
print('Bands count:', len(band_nums))
print('First/last center (Hz):', f_center_geo[0], f_center_geo[-1])
# Plot centers in Hz
plt.figure(figsize=(6,3))
plt.plot(band_nums, f_center_geo, 'o-', label='Geometric center (Hz)')
plt.xlabel('Band number')
plt.ylabel('Center frequency (Hz)')
plt.grid(True); plt.legend(); plt.tight_layout()

## Summary
- Order/cycles conversions are consistent.
- Dyadic band intervals computed from periods and frequencies behave as expected.
- Frequency bands from FFT points match the repository's test expectations.